# 01. Базовая линия: исходные потери PiSA-SR, два сида и два этапа

Исходный набор потерь PiSA-SR (L2 + 2·LPIPS + CSD), старт с релизного чекпоинта. E01 и E02 — только sem-этап с разными сидами: показывают разброс метрик, различия между методами меньше этого разброса обсуждать нельзя. E00 — оба этапа (pix, затем sem): это контроль для экспериментов E05–E15, которые тоже обучают оба этапа. E00 идёт примерно вдвое дольше E01; если сессии не хватает, запускайте этапы по отдельности (см. конец ноутбука).

## Инструкция

1. **Настройки ноутбука (панель справа).** Accelerator — `GPU T4 x2`, Internet — `On`, Persistence — `Files only`.
2. **Подключите данные (Add Input):**
   - датасет с кодом — папка `PiSA-SR-main` целиком, вместе с `freq_exp`;
   - обучающие пары LQ/HQ и валидационные пары LQ/HQ;
   - Stable Diffusion 2.1-base, релизный чекпоинт `pisa_sr.pkl` и, если используете теги, `ram_swin_large_14m.pth`.
3. **Ячейка «Пути».** Впишите свои пути. Ячейка «Проверка» сразу скажет, чего не хватает.
4. **Запускайте ячейки по порядку.** Обучение печатает строку каждые 20 шагов и валидацию каждые `EVAL_EVERY` шагов (по умолчанию 500).
5. **Время.** Сессия Kaggle ограничена; если этап не успевает, уменьшите `STEPS` или запускайте этапы по отдельности
   (см. раздел «Если сессия оборвалась» в конце).
6. **Сохраните результат.** Последняя ячейка упаковывает логи и чекпоинты в архив. Нажмите `Save Version` →
   `Save & Run All` либо скачайте архив из `/kaggle/working`. Для сравнения экспериментов
   (ноутбук `08_compare_all`) подключите сохранённые результаты как входные данные.

**Что смотреть в результатах**

- `curve.png` и `curve_*.csv` — LPIPS, VoL/GT и ошибка в высокочастотных полосах по шагам;
- `hf_err` — главная метрика: 1 = детали не восстановлены, больше 1 = выдуманы, меньше — лучше;
- `eff_gsd_cm` — эффективное разрешение в сантиметрах (меньше — лучше);
- `train_log_*.csv` — значение, вес и норма градиента каждого члена потерь. Если норма градиента частотного
  члена меньше 5 % от главного (L2 в pix-этапе, LPIPS в sem-этапе), член почти не влияет — увеличьте вес.

In [ ]:
NOTEBOOK_TAG = "01_baseline"

# ==== Пути: отредактируйте под свои датасеты ====
CODE_SRC  = "/kaggle/input/pisa-sr-main/PiSA-SR-main"      # папка с кодом (внутри должна быть freq_exp)
SD_PATH   = "/kaggle/working/sd_v2_1_full"                  # Stable Diffusion 2.1-base
INIT_CKPT = "/kaggle/input/pisa-sr-weights/pisa_sr.pkl"     # релизный чекпоинт PiSA-SR
RAM_PATH  = ""                                              # ram_swin_large_14m.pth; пусто = без RAM
TRAIN_LQ  = "/kaggle/input/hq-lq-netherlands-train/lq_images"
TRAIN_HQ  = "/kaggle/input/hq-lq-netherlands-train/hq_images"
VAL_LQ    = "/kaggle/input/datasets/vende14/hq-lq-netherlands-val/lq_images"
VAL_HQ    = "/kaggle/input/datasets/vende14/hq-lq-netherlands-val/hq_images"

# ==== Параметры обучения ====
OUT_ROOT = "/kaggle/working/experiments/freq"   # сюда пишутся логи и чекпоинты
STEPS    = 4000    # шагов на этап (как в ваших прежних запусках)
EVAL_EVERY = 500   # валидация каждые столько шагов; чаще — точнее кривая, но дольше обучение
ACCUM    = 1       # накопление градиента: 4 сглаживает кривые, но за те же шаги даёт вчетверо меньше обновлений
EMA      = 0       # 0.999 включает EMA весов LoRA (сохраняется отдельный *_ema.pkl)
FDL_CROP = 256     # окно для FDL: 256 экономит память; 0 = весь кроп 512
AUTO     = 0.3     # калибровка весов частотных членов по норме градиента; 0 = взять веса как есть

In [ ]:
import os, shutil, subprocess, sys

# --- Проверка ---
!nvidia-smi --query-gpu=name,memory.total --format=csv
missing = [p for p in [CODE_SRC, SD_PATH, INIT_CKPT, TRAIN_LQ, TRAIN_HQ, VAL_LQ, VAL_HQ] if not os.path.exists(p)]
assert not missing, f"Не найдены пути: {missing}"
assert os.path.isdir(os.path.join(CODE_SRC, "freq_exp")), "В датасете с кодом нет папки freq_exp"
if RAM_PATH and not os.path.exists(RAM_PATH):
    raise FileNotFoundError(RAM_PATH)

# --- Код копируется в рабочую папку (входные датасеты доступны только для чтения) ---
WORK = "/kaggle/working/PiSA-SR-main"
if not os.path.exists(WORK):
    shutil.copytree(CODE_SRC, WORK)
os.chdir(WORK)

# --- Пакеты: две функции потерь из статей + метрики ---
!pip install -q focal-frequency-loss fdl-pytorch lpips pyiqa
# Если окружение чистое, раскомментируйте установку зависимостей PiSA-SR:
# !pip install -q -r requirements.txt

os.environ.update(SD=SD_PATH, INIT=INIT_CKPT, RAM=RAM_PATH, TRAIN_LQ=TRAIN_LQ, TRAIN_HQ=TRAIN_HQ,
                  VAL_LQ=VAL_LQ, VAL_HQ=VAL_HQ, OUT=OUT_ROOT, STEPS=str(STEPS), ACCUM=str(ACCUM),
                  EMA=str(EMA), FDL_CROP=str(FDL_CROP), AUTO=str(AUTO), EVAL_EVERY=str(EVAL_EVERY))
print("Рабочая папка:", os.getcwd())
!bash freq_exp/experiments.sh list

In [ ]:
import glob, pandas as pd
from IPython.display import Image, display

def show(exp_prefix):
    """Кривые, последние точки валидации и нормы градиентов для эксперимента."""
    dirs = sorted(glob.glob(os.path.join(OUT_ROOT, exp_prefix + "*")))
    assert dirs, f"нет результатов для {exp_prefix}"
    d = dirs[0]; print("==", d)
    if os.path.exists(os.path.join(d, "curve.png")):
        display(Image(os.path.join(d, "curve.png"), width=700))
    cols = ["global_step", "stage", "lpips_val", "vol_ratio", "hf_frc", "hf_err", "hf_energy", "eff_gsd_cm", "mem0_gb", "mem1_gb"]
    for f in sorted(glob.glob(os.path.join(d, "curve_*.csv"))):
        c = pd.read_csv(f); display(c[cols].tail(5))
    for f in sorted(glob.glob(os.path.join(d, "train_log_*.csv"))):
        t = pd.read_csv(f).dropna(axis=1, how="all")
        g = t[[c for c in t.columns if c.startswith(("global_step", "stage", "w_", "gnorm_"))]]
        print("Веса и нормы градиентов членов потерь (последние записи):"); display(g.tail(4))

In [ ]:
# Инференс итогового чекпоинта на валидации и оценка по частотным полосам.
N_EVAL = 100          # сколько снимков (0 = все 496; дольше)
def infer(exp_prefix, ckpt_name="pisasr_final.pkl"):
    d = sorted(glob.glob(os.path.join(OUT_ROOT, exp_prefix + "*")))[0]
    out = os.path.join("/kaggle/working/sr", os.path.basename(d))
    !python freq_exp/infer_eval.py --pretrained_model_path {SD_PATH} --ckpt {d}/checkpoints/{ckpt_name} \
        --lq_dir {VAL_LQ} --hq_dir {VAL_HQ} --out_dir {out} --n {N_EVAL}
    return os.path.join(out, "lsem_1.0")

## E01. Базовая линия, сид 123

Отличие от исходного обучения PiSA-SR: `--stage sem --seed 123`

In [ ]:
!bash freq_exp/experiments.sh E01

In [ ]:
show("E01")

In [ ]:
sr_E01 = infer("E01")

## E02. Базовая линия, сид 456

Отличие от исходного обучения PiSA-SR: `--stage sem --seed 456`

In [ ]:
!bash freq_exp/experiments.sh E02

In [ ]:
show("E02")

In [ ]:
sr_E02 = infer("E02")

## E00. Контроль для E05–E15: оба этапа

Отличие от исходного обучения PiSA-SR: `--stage both --seed 123`

In [ ]:
!bash freq_exp/experiments.sh E00

In [ ]:
show("E00")

In [ ]:
sr_E00 = infer("E00")

## Разброс между сидами и вклад pix-этапа

Первая таблица (E01 против E02) — шум от сида: различия меньше него считать случайными. Вторая (E00 против E01) — разница между схемами «pix + sem» и «только sem» при тех же потерях. Эксперименты E05–E15 сравнивайте с E00, а E03–E04 — с E01.

In [ ]:
!python freq_exp/eval_bands.py --sr_dir {sr_E01} --sr_dir_b {sr_E02} --hq_dir {VAL_HQ}

In [ ]:
!python freq_exp/eval_bands.py --sr_dir {sr_E00} --sr_dir_b {sr_E01} --hq_dir {VAL_HQ}

In [ ]:
# Архив с логами и итоговыми чекпоинтами (промежуточные чекпоинты не включаются).
import zipfile
zpath = "/kaggle/working/results_" + NOTEBOOK_TAG + ".zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_ROOT):
        for fn in files:
            if fn.endswith((".csv", ".png", ".json")) or "final" in fn:
                p = os.path.join(root, fn); z.write(p, os.path.relpath(p, os.path.dirname(OUT_ROOT)))
    if os.path.isdir("/kaggle/working/sr"):
        for root, _, files in os.walk("/kaggle/working/sr"):
            for fn in files:
                if fn.endswith(".csv"):
                    p = os.path.join(root, fn); z.write(p, os.path.relpath(p, "/kaggle/working"))
print(zpath, round(os.path.getsize(zpath) / 2**20, 1), "МБ")

## Если сессия оборвалась

- **Оба этапа не помещаются в одну сессию.** Запустите этапы по отдельности тем же набором флагов:
  сначала `--stage pix`, затем `--stage sem` (он возьмёт `checkpoints/pisasr_pix_final.pkl` из той же папки
  результата). Команду со всеми флагами печатает `experiments.sh` в строке `=== ...` перед запуском;
  замените в ней `--stage both` на нужный этап.
- **Не хватает памяти (CUDA out of memory).** Проверьте, что выбраны две видеокарты; оставьте `FDL_CROP = 256`;
  в крайнем случае добавьте к команде `--fdl_num_proj 64`.
- **Потеря стала NaN.** Чаще всего это Fourier Space Loss со слишком большим весом: оставьте `AUTO = 0.3`.